In [1]:
import keras
import numpy as np

Using TensorFlow backend.


Word-level onehot encoding

In [2]:
#Build a samples
#in this case is a single sentences but it can be a text
samples = ['We are the only sane person in this crazy world.',
           'You know, the moral code, is a bull shit',
           ' They only keep it when the circumstances allow them to do that']

In [6]:
token_index = {}
for sample in samples:
    for word in sample.split():
        if word not in token_index:
            token_index[word] = len(token_index) + 1
max_length = 10

results = np.zeros(shape = (len(samples), 
                              max_length, 
                              max(token_index.values()) + 1))
for i, sample in enumerate(samples):
    for j, word in list(enumerate(sample.split()))[:max_length]:
        index = token_index.get(word)
        results[i,j, index] = 1

In [7]:
results

array([[[ 0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
          0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
          0.,  0.,  0.],
        [ 0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
          0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
          0.,  0.,  0.],
        [ 0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
          0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
          0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
          0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
          0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
          0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
          0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,
          0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0

Using Keras for word-level one hot encoding

In [8]:
from keras.preprocessing.text import Tokenizer
samples = ['The guy sitting next to me may be crazy',
           'He keep typing quickly without notice anthing, and sometimes he looks into others computer']

In [10]:
tokenizer = Tokenizer(num_words = 1000)
tokenizer.fit_on_texts(samples)
sequences = tokenizer.texts_to_sequences(samples)
one_hot_results = tokenizer.texts_to_matrix(samples, mode = 'binary')
word_index = tokenizer.word_index

In [11]:
len(word_index)

22

In [12]:
word_index

{'and': 17,
 'anthing': 16,
 'be': 9,
 'computer': 22,
 'crazy': 10,
 'guy': 3,
 'he': 1,
 'into': 20,
 'keep': 11,
 'looks': 19,
 'may': 8,
 'me': 7,
 'next': 5,
 'notice': 15,
 'others': 21,
 'quickly': 13,
 'sitting': 4,
 'sometimes': 18,
 'the': 2,
 'to': 6,
 'typing': 12,
 'without': 14}

In [13]:
sequences

[[2, 3, 4, 5, 6, 7, 8, 9, 10],
 [1, 11, 12, 13, 14, 15, 16, 17, 18, 1, 19, 20, 21, 22]]

In [14]:
one_hot_results

array([[ 0.,  0.,  1., ...,  0.,  0.,  0.],
       [ 0.,  1.,  0., ...,  0.,  0.,  0.]])

# Using embedding word

In [2]:
from keras.datasets import imdb
from keras import preprocessing

max_features = 100000 #number of words to considers as features
max_len = 50 # cut off the text of this number of word
(x_train, y_train), (x_test, y_test )= imdb.load_data(num_words = max_features)

In [ ]:
x_train[0]

In [8]:
#preprocess the data
#turn th list of integers into a 2D intiger tensor of shape (samples, max_len)
x_train = preprocessing.sequence.pad_sequences(x_train, maxlen = max_len)
x_test = preprocessing.sequence.pad_sequences(x_test, maxlen = max_len)

In [9]:
x_train[0]

array([2071,   56,   26,  141,    6,  194, 7486,   18,    4,  226,   22,
         21,  134,  476,   26,  480,    5,  144,   30, 5535,   18,   51,
         36,   28,  224,   92,   25,  104,    4,  226,   65,   16,   38,
       1334,   88,   12,   16,  283,    5,   16, 4472,  113,  103,   32,
         15,   16, 5345,   19,  178,   32])

In [15]:
#using an embedding layer and classifier on the IMDB data
from keras.models import Sequential
from keras.layers import Flatten, Dense
from keras.layers import Embedding

embedding_layers = Embedding(1000,64)
model = Sequential()
model.add(Embedding(max_features, 8, input_length = max_len))
model.add(Flatten())
model.add(Dense(1, activation = 'sigmoid'))
model.compile(optimizer = 'rmsprop', loss = 'binary_crossentropy', metrics = ['acc'])
model.summary()

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_5 (Embedding)      (None, 50, 8)             800000    
_________________________________________________________________
flatten_3 (Flatten)          (None, 400)               0         
_________________________________________________________________
dense_3 (Dense)              (None, 1)                 401       
Total params: 800,401
Trainable params: 800,401
Non-trainable params: 0
_________________________________________________________________


In [16]:
#train the model
history = model.fit(x_train, y_train,
                   epochs = 10, 
                   batch_size = 32, 
                   validation_split = 0.2)

Train on 20000 samples, validate on 5000 samples
Epoch 1/10
20000/20000 [==============================] - 11s 558us/step - loss: 0.6526 - acc: 0.6469 - val_loss: 0.5552 - val_acc: 0.7584
Epoch 2/10
20000/20000 [==============================] - 9s 452us/step - loss: 0.4511 - acc: 0.8053 - val_loss: 0.4310 - val_acc: 0.7974
Epoch 3/10
20000/20000 [==============================] - 10s 481us/step - loss: 0.3560 - acc: 0.8473 - val_loss: 0.4031 - val_acc: 0.8120
Epoch 4/10
20000/20000 [==============================] - 10s 485us/step - loss: 0.3131 - acc: 0.8666 - val_loss: 0.3986 - val_acc: 0.8154
Epoch 5/10
20000/20000 [==============================] - 9s 472us/step - loss: 0.2844 - acc: 0.8808 - val_loss: 0.4002 - val_acc: 0.8170
Epoch 6/10
20000/20000 [==============================] - 10s 505us/step - loss: 0.2613 - acc: 0.8921 - val_loss: 0.4035 - val_acc: 0.8190
Epoch 7/10
20000/20000 [==============================] - 9s 430us/step - loss: 0.2408 - acc: 0.9032 - val_loss: 0.4102